# Free-convection LES example

This notebook compares the final official LES profiles with OceanTurb CATKE, KPP, MY2.5, $k$-$\omega$, and $k$-$\epsilon$. All OceanTurb integrations start from the same regridded LES buoyancy and passive-tracer profiles at $t=600$ s and use a common 10 s time step.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

experiment_dir = Path.cwd()
output_dir = experiment_dir / "output" / "dt10s"
figure_dir = experiment_dir / "figures"
figure_dir.mkdir(exist_ok=True)

suites = [
    (6, "extreme forcing"),
    (12, "strong forcing"),
    (24, "medium forcing"),
    (48, "weak forcing"),
    (72, "very weak forcing"),
]
profiles = {
    hours: pd.read_csv(output_dir / f"free_convection_{hours:02d}h.csv")
    for hours, _ in suites
}

list(profiles[6].columns)

In [ ]:
styles = [
    ("catke", "CATKE", "#111111", "-", 2.0),
    ("kpp", "KPP", "#2878B5", "--", 2.0),
    ("my25", "MY2.5", "#E09F28", "-.", 2.0),
    ("komega", r"$k$-$\omega$", "#D64B3C", ":", 2.2),
    ("kepsilon", r"$k$-$\epsilon$", "#8055A6", (0, (6, 2)), 2.0),
]

def draw_profiles(y_limits, filename, title_suffix):
    fig, axes = plt.subplots(2, 5, figsize=(17.0, 8.0), sharey=False)

    for column, (hours, forcing_label) in enumerate(suites):
        data = profiles[hours]
        z = data["z_m"]
        b_reference = data.loc[27, "initial_les_buoyancy_m_s-2"]
        ax_b, ax_c = axes[0, column], axes[1, column]

        ax_b.plot(
            1e4 * (data["final_les_buoyancy_m_s-2"] - b_reference),
            z,
            color="seagreen",
            alpha=0.55,
            linewidth=6.0,
            label="LES",
            solid_capstyle="round",
        )
        ax_c.plot(
            data["final_les_passive_tracer"],
            z,
            color="seagreen",
            alpha=0.55,
            linewidth=6.0,
            label="LES",
            solid_capstyle="round",
        )

        for key, label, color, linestyle, linewidth in styles:
            ax_b.plot(
                1e4 * (data[f"{key}_buoyancy_m_s-2"] - b_reference),
                z,
                color=color,
                linestyle=linestyle,
                linewidth=linewidth,
                label=label,
            )
            ax_c.plot(
                data[f"{key}_passive_tracer"],
                z,
                color=color,
                linestyle=linestyle,
                linewidth=linewidth,
            )

        ax_b.set_title(
            f"{hours} hour simulation\n({forcing_label})", fontsize=11
        )
        ax_b.set_xlabel("Buoyancy\n($10^{-4}$ m s$^{-2}$)")
        ax_c.set_xlabel("Passive tracer")

        panel_limits = y_limits[column] if isinstance(y_limits, list) else y_limits
        for ax in (ax_b, ax_c):
            ax.set_ylim(*panel_limits)
            ax.margins(x=0.07)
            ax.grid(alpha=0.18)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            if column == 0:
                ax.set_ylabel("z (m)")
            else:
                ax.tick_params(labelleft=False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=6,
        frameon=False,
        bbox_to_anchor=(0.5, 0.925),
    )
    fig.suptitle(
        "Free convection: OceanTurb closures compared with LES"
        f"\nCommon initialization at 10 minutes and $\\Delta t=10$ s{title_suffix}",
        y=0.985,
        fontsize=15,
    )
    fig.subplots_adjust(
        left=0.055, right=0.985, bottom=0.07, top=0.84,
        wspace=0.18, hspace=0.38,
    )
    path = figure_dir / filename
    fig.savefig(path, dpi=240, bbox_inches="tight")
    plt.close(fig)
    return path

full_depth_path = draw_profiles(
    (-256, 0),
    "free_convection_closures_full_depth.png",
    " — full depth",
)
full_depth_path

In [ ]:
# A second version uses panel-dependent upper-ocean limits.
# Horizontal limits remain data-driven so no closure profile is clipped.
vertical_zoom_path = draw_profiles(
    [(-150, 5), (-160, 5), (-165, 5), (-180, 5), (-180, 5)],
    "free_convection_closures.png",
    " — upper-ocean vertical zoom",
)
vertical_zoom_path

In [ ]:
summary = pd.read_csv(output_dir / "run_summary.txt", skiprows=8)
summary